# Learning `pytest` with the RINGSS pipeline — Week 38

**A teaching notebook.** By the end you'll understand what `pytest` is, how it
finds and runs tests, and how to write tests for numeric/scientific code like
the RINGSS simulation. Every code cell below runs **real pytest** and the
markdown above it explains what you're about to see.

### How this notebook runs pytest
Normally pytest runs from a terminal on files named `test_*.py`. To learn it
*interactively*, the first code cell defines a tiny helper `run_pytest(...)`
that writes a lesson's tests to a temporary `test_*.py` file and runs genuine
pytest on it — so the output you see is exactly what you'd get on the command
line. (In real projects your tests live in `test_*.py` files; in notebooks
people often use the `ipytest` package — we avoid that extra dependency here.)

> Read the output of each cell! The line like `1 passed in 0.01s` (green) or
> `1 failed` (red) is pytest's verdict. A couple of cells **fail on purpose** —
> that's the lesson.

## 0. The in-notebook pytest runner

Run this first. It defines `run_pytest(code)` and immediately proves it works
with one trivial test. Notice pytest reports `test_the_runner_works PASSED`.

In [1]:
# === In-notebook pytest runner ==========================================
import pytest, tempfile, textwrap, shutil, pathlib

_lesson = 0
def run_pytest(code, *args):
    """Write `code` to a temp test_*.py file and run REAL pytest on it (-v)."""
    global _lesson
    _lesson += 1
    src = textwrap.dedent(code).strip("\n") + "\n"
    d = pathlib.Path(tempfile.mkdtemp(prefix="nb_pytest_"))
    f = d / f"test_lesson_{_lesson}.py"
    f.write_text(src)
    try:
        # -v = verbose (one line per test); importlib mode lets us reuse names;
        # no:cacheprovider keeps the folder clean (no .pytest_cache).
        return pytest.main([str(f), "-v", "-p", "no:cacheprovider",
                            "--import-mode=importlib", *args])
    finally:
        shutil.rmtree(d, ignore_errors=True)

# Smoke-test the runner:
run_pytest("""
    def test_the_runner_works():
        assert True
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_x_nvywkj
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 1 item

../../../../../../../../../../tmp/nb_pytest_x_nvywkj/test_lesson_1.py::test_the_runner_works PASSED [100%]

============================== 1 passed in 0.01s ===============================


<ExitCode.OK: 0>

## 1. Your first test: `assert` + naming rules

pytest's core is dead simple: **a test is a function whose name starts with
`test_`, and it uses the plain Python `assert` statement.** If every `assert`
is true, the test *passes*; if any raises `AssertionError`, it *fails*.

pytest **discovers** tests automatically by these conventions:
- files named `test_*.py` or `*_test.py`
- functions named `test_*`
- classes named `Test*` (with `test_*` methods)

You don't call the test functions yourself — pytest collects and runs them.

In [5]:
run_pytest("""
    def test_one_plus_one():
        assert 1 + 1 == 2

    def test_string_method():
        assert "ring".upper() == "RING"

    def test_membership():
        assert 3 in [1, 2, 3]
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_ld9rsb97
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 3 items

../../../../../../../../../../tmp/nb_pytest_ld9rsb97/test_lesson_5.py::test_one_plus_one PASSED [ 33%]
../../../../../../../../../../tmp/nb_pytest_ld9rsb97/test_lesson_5.py::test_string_method PASSED [ 66%]
../../../../../../../../../../tmp/nb_pytest_ld9rsb97/test_lesson_5.py::test_membership PASSED [100%]

============================== 3 passed in 0.01s ===============================


<ExitCode.OK: 0>

## 2. What a failure looks like (assert introspection)

pytest's superpower: when an `assert` fails it **rewrites** it to show you the
actual values — you don't need `assert x == y, "message"`. 

**The next cell fails on purpose.** Look for:
- the red `F` and `FAILED`,
- the line `assert result == 5`,
- pytest printing `where result = 4` (or `4 == 5`) — that's the introspection
  telling you *why* it failed. This is what makes debugging fast.

In [6]:
run_pytest("""
    def test_this_fails_on_purpose():
        result = 2 + 2
        assert result == 5    # WRONG (result is 4) -- pytest shows both values
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_rn8pjz3j
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 1 item

../../../../../../../../../../tmp/nb_pytest_rn8pjz3j/test_lesson_6.py::test_this_fails_on_purpose FAILED [100%]

=================================== FAILURES ===================================
__________________________ test_this_fails_on_purpose __________________________

    def test_this_fails_on_purpose():
        result = 2 + 2
>       assert result == 5    # WRONG (result is 4) -- pytest shows both values
        ^^^^^^^^^^^^^^^^^^
E       assert 4 == 5

/tmp/nb_pytest_rn8pjz3j/test_lesson_6.py:3: AssertionError
=========================== short test summary inf

<ExitCode.TESTS_FAILED: 1>

## 3. Floating-point: never use `==`, use `pytest.approx`

RINGSS is all floating-point. Because of rounding, `0.1 + 0.2 == 0.3` is
`False` in Python! Comparing floats with `==` gives fragile, flaky tests.
Use **`pytest.approx`**, which compares within a tolerance:
- `x == pytest.approx(y)` — default relative tolerance 1e-6
- `pytest.approx(y, abs=1e-4)` — absolute tolerance
- `pytest.approx(y, rel=1e-3)` — relative tolerance

Here we test the **Fried parameter** `r0 = 0.98·λ / seeing` from your `sim1.par`
cell.

In [7]:
run_pytest("""
    import pytest

    def fried_r0(seeing_arcsec, wavelen_m):
        # seeing is in arcsec -> convert to radians with /206265
        return 0.98 * wavelen_m / (seeing_arcsec / 206265.0)

    def test_float_equality_trap():
        assert 0.1 + 0.2 != 0.3                      # the trap: True!
        assert 0.1 + 0.2 == pytest.approx(0.3)       # the fix

    def test_fried_r0_value():
        r0 = fried_r0(1.0, 0.6e-6)                    # 1 arcsec seeing
        assert r0 == pytest.approx(0.1213, abs=1e-4)
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_5wjonh_m
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 2 items

../../../../../../../../../../tmp/nb_pytest_5wjonh_m/test_lesson_7.py::test_float_equality_trap PASSED [ 50%]
../../../../../../../../../../tmp/nb_pytest_5wjonh_m/test_lesson_7.py::test_fried_r0_value PASSED [100%]

============================== 2 passed in 0.01s ===============================


<ExitCode.OK: 0>

## 4. Testing that errors happen: `pytest.raises`

Sometimes correct behaviour *is* raising an error. Your `simatm` cell does
`if zlow >= zhigh: raise ValueError(...)`. To assert an exception is raised,
wrap the call in a `with pytest.raises(...)` block. The test passes **only if**
that exception is raised. You can also check the message with `match=` (a regex).

In [9]:
run_pytest("""
    import pytest

    def check_layers(zlow, zhigh):
        if zlow >= zhigh:
            raise ValueError(f"zlow ({zlow}) must be below zhigh ({zhigh})")
        return True

    def test_valid_layers_ok():
        assert check_layers(500, 10500) is True

    def test_bad_layers_raise():
        with pytest.raises(ValueError):
            check_layers(10500, 500)

    def test_error_message_matches():
        with pytest.raises(ValueError, match="must be below"):
            check_layers(500, 500)
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_gsq1gimv
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 3 items

../../../../../../../../../../tmp/nb_pytest_gsq1gimv/test_lesson_9.py::test_valid_layers_ok PASSED [ 33%]
../../../../../../../../../../tmp/nb_pytest_gsq1gimv/test_lesson_9.py::test_bad_layers_raise PASSED [ 66%]
../../../../../../../../../../tmp/nb_pytest_gsq1gimv/test_lesson_9.py::test_error_message_matches PASSED [100%]

============================== 3 passed in 0.01s ===============================


<ExitCode.OK: 0>

## 5. `@pytest.mark.parametrize`: one test, many cases

Instead of copy-pasting a test for each input, **parametrize** it: pytest runs
the function once per tuple and reports each case separately (so you see exactly
which input failed). Below we test that the ring radius scales as `1/pdist`, and
that the invariant `H·R = radius × pdist` stays constant (the thing your
`getweight5` relies on to back-solve `pdist`).

In [10]:
run_pytest("""
    import pytest

    def ring_radius_px(d, eps, pdist, platescale_arcsec):
        return 0.85 * d * (1 + eps) / (4 * pdist) * 206265 / platescale_arcsec

    @pytest.mark.parametrize("pdist, expected_px", [
        (1200, 16.03),
        (1050, 18.32),
        (600,  32.06),
    ])
    def test_ring_radius_values(pdist, expected_px):
        r = ring_radius_px(0.304, 0.7, pdist, 1.1777)
        assert r == pytest.approx(expected_px, abs=0.05)

    @pytest.mark.parametrize("pdist", [600, 1050, 1200])
    def test_HR_invariant(pdist):
        r = ring_radius_px(0.304, 0.7, pdist, 1.1777)
        assert r * pdist == pytest.approx(19234, rel=1e-3)  # H*R is constant
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_tsu4mpjx
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 6 items

../../../../../../../../../../tmp/nb_pytest_tsu4mpjx/test_lesson_10.py::test_ring_radius_values[1200-16.03] PASSED [ 16%]
../../../../../../../../../../tmp/nb_pytest_tsu4mpjx/test_lesson_10.py::test_ring_radius_values[1050-18.32] PASSED [ 33%]
../../../../../../../../../../tmp/nb_pytest_tsu4mpjx/test_lesson_10.py::test_ring_radius_values[600-32.06] PASSED [ 50%]
../../../../../../../../../../tmp/nb_pytest_tsu4mpjx/test_lesson_10.py::test_HR_invariant[600] PASSED [ 66%]
../../../../../../../../../../tmp/nb_pytest_tsu4mpjx/test_lesson_10.py::test_HR_invariant[1050] PASSE

<ExitCode.OK: 0>

## 6. Fixtures: reusable setup

A **fixture** is a function decorated with `@pytest.fixture` that builds
something your tests need (data, a config, a temp file). A test *requests* a
fixture simply by naming it as a parameter — pytest calls the fixture and
injects the result. This keeps setup in one place and out of the tests.

Here a fixture builds a synthetic 64×64 ring image; two tests consume it.

In [11]:
run_pytest("""
    import numpy as np
    import pytest

    @pytest.fixture
    def ring_image():
        \"\"\"A 64x64 synthetic annulus centred in the frame.\"\"\"
        n = 64
        y, x = np.ogrid[:n, :n]
        r = np.hypot(x - n / 2, y - n / 2)
        return ((r > 14) & (r < 18)).astype(float)

    def test_ring_has_pixels(ring_image):
        assert ring_image.sum() > 0

    def test_ring_is_centred(ring_image):
        n = ring_image.shape[0]
        ys, xs = np.nonzero(ring_image)
        assert xs.mean() == pytest.approx(n / 2, abs=0.5)
        assert ys.mean() == pytest.approx(n / 2, abs=0.5)
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_6qp89o_o
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 2 items

../../../../../../../../../../tmp/nb_pytest_6qp89o_o/test_lesson_11.py::test_ring_has_pixels PASSED [ 50%]
../../../../../../../../../../tmp/nb_pytest_6qp89o_o/test_lesson_11.py::test_ring_is_centred PASSED [100%]

============================== 2 passed in 0.01s ===============================


<ExitCode.OK: 0>

## 7. Testing NumPy arrays: `np.testing`

`assert arr1 == arr2` on arrays raises *"truth value of an array is ambiguous"* —
because `==` returns an array, not a bool. For arrays use **`numpy.testing`**:
- `np.testing.assert_allclose(a, b, rtol=...)` — approx-equal, elementwise
- `np.testing.assert_array_equal(a, b)` — exactly equal

We test your **flux-conserving rebin** (the `zoom(...)*(nap/nccd)**2` trick from
the fixed `ringsim` cell): total flux must be preserved and the shape correct.

In [12]:
run_pytest("""
    import numpy as np
    from scipy.ndimage import zoom

    def flux_conserving_rebin(img, nout):
        nin = img.shape[0]
        return zoom(img, nout / nin, order=1) * (nin / nout) ** 2

    def test_rebin_output_shape():
        out = flux_conserving_rebin(np.ones((64, 64)), 16)
        assert out.shape == (16, 16)

    def test_rebin_conserves_flux_exactly_for_constant():
        out = flux_conserving_rebin(np.ones((64, 64)), 16)
        np.testing.assert_allclose(out.sum(), 64 * 64, rtol=1e-9)

    def test_rebin_conserves_flux_approx_for_random():
        rng = np.random.default_rng(0)
        img = rng.random((64, 64))
        out = flux_conserving_rebin(img, 16)
        np.testing.assert_allclose(out.sum(), img.sum(), rtol=1e-2)
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_6svmn_zr
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 3 items

../../../../../../../../../../tmp/nb_pytest_6svmn_zr/test_lesson_12.py::test_rebin_output_shape PASSED [ 33%]
../../../../../../../../../../tmp/nb_pytest_6svmn_zr/test_lesson_12.py::test_rebin_conserves_flux_exactly_for_constant PASSED [ 66%]
../../../../../../../../../../tmp/nb_pytest_6svmn_zr/test_lesson_12.py::test_rebin_conserves_flux_approx_for_random PASSED [100%]

============================== 3 passed in 0.15s ===============================


<ExitCode.OK: 0>

## 8. Marks: `skip`, `skipif`, `xfail`

**Marks** annotate tests:
- `@pytest.mark.skip(reason=...)` — never run (e.g. a feature not built yet, like
  the standalone `ringsim.py`).
- `@pytest.mark.skipif(condition, reason=...)` — run only if condition is False.
- `@pytest.mark.xfail(reason=...)` — you *expect* this to fail (a known bug);
  pytest reports it as `xfailed` (not a red failure) and `XPASS` if it
  unexpectedly starts passing.

In the output you'll see `s` (skipped) and `x` (xfailed) instead of `.`.

In [13]:
run_pytest("""
    import sys, pytest

    @pytest.mark.skip(reason="standalone ringsim.py not implemented yet")
    def test_future_feature():
        assert False              # never runs, so never fails

    @pytest.mark.skipif(sys.version_info < (3, 8), reason="needs Python >= 3.8")
    def test_runs_on_modern_python():
        assert True

    @pytest.mark.xfail(reason="known floating-point edge case, documented")
    def test_known_bug():
        assert 0.1 + 0.2 == 0.3   # expected to fail -> reported as xfail
""")

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.1.1, pluggy-1.6.0 -- /opt/lsst/software/stack/conda/envs/lsst-scipipe-12.3.0-exact/bin/python3
rootdir: /tmp/nb_pytest_nz664c11
plugins: anyio-4.14.2, cov-7.1.0, doctestplus-1.7.1, session2file-0.1.11, vcr-1.0.2, xdist-3.8.0, typeguard-4.6.0, nbval-0.11.0, zarr-3.3.0
collecting ... collected 3 items

../../../../../../../../../../tmp/nb_pytest_nz664c11/test_lesson_13.py::test_future_feature SKIPPED [ 33%]
../../../../../../../../../../tmp/nb_pytest_nz664c11/test_lesson_13.py::test_runs_on_modern_python PASSED [ 66%]
../../../../../../../../../../tmp/nb_pytest_nz664c11/test_lesson_13.py::test_known_bug XFAIL [100%]

=================== 1 passed, 1 skipped, 1 xfailed in 0.03s ====================


<ExitCode.OK: 0>

## 9. A mini RINGSS suite: classes + fixtures + parametrize together

Real test files combine everything. You can group related tests in a class named
`Test*` (purely for organisation — no `self` setup needed). Here we verify the
**seeing ⇄ turbulence-integral round trip**: derive `r0` from a seeing value,
compute the integral `J`, convert `J` back to seeing, and check we recover the
input.

Teaching subtlety: the constant `6.83e-13` is defined **at 500 nm**, so the
round trip only closes cleanly when `λ = 0.5 µm`. Using it at 0.6 µm would drift
~4% — a great example of a test encoding a physical assumption.

In [ ]:
run_pytest("""
    import numpy as np
    import pytest

    SEECONST = 6.83e-13   # turbulence integral for 1-arcsec seeing at 500 nm

    def turbulence_integral(r0, wavelen):
        return (r0 ** (-5 / 3)) / 0.423 * ((0.5 * wavelen / np.pi) ** 2)

    def seeing_from_integral(j):
        return (j / SEECONST) ** 0.6

    @pytest.fixture
    def wavelen():
        return 0.5e-6          # 500 nm, matching SEECONST's reference

    class TestSeeingRoundTrip:
        @pytest.mark.parametrize("seeing_in", [0.5, 1.0, 1.5, 2.0])
        def test_roundtrip_recovers_input(self, seeing_in, wavelen):
            r0 = 0.98 * wavelen / (seeing_in / 206265.0)
            j = turbulence_integral(r0, wavelen)
            assert seeing_from_integral(j) == pytest.approx(seeing_in, rel=1e-3)

        def test_more_turbulence_is_worse_seeing(self, wavelen):
            j_good = turbulence_integral(0.20, wavelen)   # big r0 = calm
            j_bad  = turbulence_integral(0.05, wavelen)   # small r0 = turbulent
            assert j_bad > j_good
""")

## 10. How you run pytest *for real* (outside a notebook)

In a real project you don't use a helper — you put tests in `test_*.py` files and
run pytest from the terminal:

```
pytest                    # discover & run everything under the current folder
pytest -v                 # verbose: one line per test
pytest test_ring.py       # just one file
pytest -k "radius"        # only tests whose name matches "radius"
pytest -x                 # stop at the first failure
pytest --maxfail=2        # stop after 2 failures
pytest -q                 # quiet
```

Useful extras: a `conftest.py` holds fixtures shared across many test files;
`pytest.ini`/`pyproject.toml` configures defaults. The cell below writes a real
`test_ring_demo.py` and invokes `python -m pytest` on it via `subprocess`, so you
see the authentic command-line experience (then cleans the file up).

In [ ]:
import subprocess, sys, textwrap, pathlib

p = pathlib.Path("test_ring_demo.py")
p.write_text(textwrap.dedent("""
    def ring_radius_px(d, eps, pdist, ps):
        return 0.85 * d * (1 + eps) / (4 * pdist) * 206265 / ps

    def test_radius_is_in_expected_ballpark():
        assert 15 < ring_radius_px(0.304, 0.7, 1200, 1.1777) < 17
"""))
result = subprocess.run([sys.executable, "-m", "pytest", "test_ring_demo.py", "-v"],
                        capture_output=True, text=True)
print(result.stdout)
p.unlink()   # clean up the demo file

## 11. Recap & how to test *your* pipeline next

**Cheat-sheet**
| Concept | Syntax |
|---|---|
| a test | `def test_x(): assert ...` |
| float compare | `x == pytest.approx(y, abs=..., rel=...)` |
| expect an error | `with pytest.raises(ValueError, match="..."):` |
| many cases | `@pytest.mark.parametrize("a,b", [(...), (...)])` |
| shared setup | `@pytest.fixture` + request by parameter name |
| arrays | `np.testing.assert_allclose(a, b, rtol=...)` |
| skip / known-fail | `@pytest.mark.skip / .skipif / .xfail` |
| run it | `pytest -v` in the terminal |

**Applying this to RINGSS.** The pipeline logic currently lives inside notebook
cells, which aren't importable by a test file. The high-value next step is to
extract the *pure* functions (e.g. `fried_r0`, `ring_radius_px`,
`turbulence_integral`, `flux_conserving_rebin`, `blackbody`) into a module such
as `ringss_lib.py`, then write a `test_ringss.py` next to it:

```python
from ringss_lib import fried_r0, ring_radius_px
import pytest

def test_r0(): assert fried_r0(1.0, 0.6e-6) == pytest.approx(0.1213, abs=1e-4)
```

Good things to assert for scientific code: **known analytic values**,
**invariants** (like `H·R = const`), **conservation laws** (flux, normalisation),
**monotonic trends** (more turbulence → worse seeing), **round trips**
(forward then inverse returns the input), and **shapes/dtypes** of arrays.
These catch regressions without needing a "true" reference image.